# Mini-Projet NLP : Transformers et Système RAG

Ce notebook présente une exploration pratique des modèles **Transformers** pour différentes tâches de traitement automatique du langage naturel, puis l’implémentation d’un système **RAG** simple permettant d’améliorer les réponses d’un modèle de langage à l’aide d’un corpus externe.

Le projet est divisé en deux grandes parties :

1. Exploration des Transformers et tâches NLP.
2. Systèmes RAG (Retrieval-Augmented Generation).

## Environnement et bibliothèques utilisées

Dans ce notebook, nous utiliserons principalement les bibliothèques suivantes :

* `transformers` : pour charger et utiliser des modèles pré-entraînés.
* `sentence-transformers` : pour générer les embeddings des documents et des requêtes.
* `faiss` : pour indexer les embeddings et effectuer une recherche de similarité.
* `torch` : pour l’exécution des modèles de deep learning.
* `pandas` et `numpy` : pour la manipulation des données.
* `PyPDF2` ou `pypdf` : pour lire le contenu des documents PDF si nécessaire.

Ces bibliothèques permettent de construire une chaîne complète allant du traitement du texte jusqu’à la génération de réponses augmentées par des documents externes.

## Partie 1 : Exploration des Transformers et tâches NLP

Les Transformers sont des architectures de deep learning très utilisées en NLP.  
Ils permettent de traiter du texte en capturant les relations entre les mots grâce au mécanisme d’attention.

Dans cette partie, nous allons utiliser des modèles pré-entraînés disponibles sur HuggingFace afin de réaliser plusieurs tâches NLP sans entraîner un modèle depuis zéro.

In [ ]:
# Installation des bibliothèques nécessaires
!pip install datasets accelerate
!pip install -q transformers==4.44.2

## 1-a Importation des bibliothèques

In [19]:
import torch
from transformers import AutoTokenizer, AutoModel, pipeline

device = 0 if torch.cuda.is_available() else -1

print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

## 1-b Exploration du tokenizer et du modèle Transformer

Avant d’utiliser les pipelines simplifiés, il est important de comprendre comment un modèle Transformer reçoit le texte.

Un modèle Transformer ne comprend pas directement les phrases sous forme de texte brut.

Le texte passe d’abord par un tokenizer :

Texte brut → Tokens → Identifiants numériques → Modèle Transformer

Le tokenizer découpe le texte en mots ou sous-mots, puis transforme ces éléments en nombres appelés `input_ids`.

In [20]:
from transformers import AutoTokenizer, AutoModel

# Modèle BERT adapté au français
model_name = "dbmdz/bert-base-french-europeana-cased"

# Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Chargement du modèle
model = AutoModel.from_pretrained(model_name)

# Exemple de phrase
texte = "Les architectures Transformers ont révolutionné le traitement du langage naturel."

# Encodage du texte
inputs = tokenizer(texte, return_tensors="pt")

print("Texte original :")
print(texte)

print("\nTokens générés :")
print(tokenizer.tokenize(texte))

print("\nIdentifiants numériques des tokens :")
print(inputs["input_ids"])

# Passage dans le modèle
outputs = model(**inputs)

print("\nForme de la dernière couche cachée :")
print(outputs.last_hidden_state.shape)

## 1-c-0 Utilisation des pipelines HuggingFace

HuggingFace propose une fonction appelée `pipeline`.

Cette fonction permet d’utiliser facilement des modèles pré-entraînés pour différentes tâches NLP, sans écrire manuellement toutes les étapes internes.

Dans cette partie, nous testons quatre tâches :

1. Analyse de sentiment
2. Classification de texte
3. Question Answering
4. Résumé automatique

## 1-c-1 Analyse de sentiment

L’analyse de sentiment consiste à déterminer l’opinion exprimée dans un texte.

Le résultat peut être positif, négatif ou parfois neutre selon le modèle utilisé.

Exemple :

Texte : "Ce projet est intéressant."  
Résultat attendu : sentiment positif

In [21]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=device
)

textes = [
    "Ce film est absolument magnifique !",
    "Je n'ai pas du tout aimé cette expérience.",
]

for texte in textes:
    result = sentiment_pipeline(texte)
    print(f"Texte   : {texte}")
    print(f"Sentiment label : {result[0]['label']} , score : {result[0]['score']:.4f}\n")


### Analyse du résultat

Le modèle retourne généralement deux informations :

- `label` : la classe prédite, par exemple positif ou négatif.
- `score` : le niveau de confiance du modèle.

Plus le score est proche de 1, plus le modèle est confiant dans sa prédiction.

## 1-c-2 Classification de texte

La classification de texte consiste à attribuer une catégorie à un texte.

Dans cette expérience, nous utilisons la classification zero-shot.

Le zero-shot classification permet de classer un texte dans des catégories choisies sans entraîner un nouveau modèle.

In [22]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

texte_a_classer = "L'équipe de développement a déployé une nouvelle base de données vectorielle ce matin."

labels_possibles = [
    "informatique",
    "sport",
    "économie",
    "politique"
]

resultat_classification = classifier(
    texte_a_classer,
    candidate_labels=labels_possibles
)

print("--- Classification de Texte ---")
print("Texte à classer : ",texte_a_classer)

print("\nRésultats :")
for label, score in zip(resultat_classification["labels"], resultat_classification["scores"]):
    print(f"{label} : {score:.4f}")

### Analyse du résultat

Le modèle compare le texte avec chaque label proposé.

Il retourne les labels triés selon leur probabilité.

Dans notre exemple, le label le plus logique devrait être `informatique`, car le texte parle de développement et de base de données vectorielle.

## 1-c-3 Question Answering

Le Question Answering consiste à répondre à une question à partir d’un contexte donné.

Le modèle ne répond pas librement : il cherche la réponse dans le texte fourni.

In [25]:
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/xlm-roberta-base-squad2",
    device=device
)

contexte = """
L'École Nationale des Sciences Appliquées d'Al Hoceima propose une filière en Ingénierie des Données.
Au cours de la deuxième année, les étudiants réalisent un mini-projet portant sur l'exploration des modèles Transformers
et la mise en œuvre de systèmes de génération augmentée par récupération, appelés systèmes RAG.
"""

question = "Quel est le sujet du mini-projet ?"

reponse = qa_pipeline(
    question=question,
    context=contexte
)

print("Question :", question)
print("Réponse extraite :", reponse["answer"])
print("Score de confiance :", round(reponse["score"], 4))

### Analyse du résultat

Le modèle lit le contexte fourni, puis extrait la partie du texte qui répond le mieux à la question.

Le score indique la confiance du modèle.

Cette tâche montre que le modèle peut répondre correctement lorsqu’on lui donne un contexte pertinent.

## 1-c-4 Résumé automatique

Le résumé automatique consiste à produire une version courte d’un texte long.

L’objectif est de conserver les idées principales tout en réduisant la taille du texte.

Cette tâche est utile pour :

- Résumer des articles.
- Résumer des documents PDF.
- Extraire les informations importantes d’un corpus.

In [27]:
summarizer = pipeline(
    "summarization",
    model="lincoln/mbart-mlsum-automatic-summarization",
    device=device
)
texte_long = """
Le traitement du langage naturel a franchi un cap important avec l'introduction de l'architecture Transformer en 2017.
Contrairement aux réseaux récurrents traditionnels, qui traitent les mots de manière séquentielle,
les Transformers s'appuient sur un mécanisme d'attention. Ce mécanisme permet au modèle d'analyser les relations
entre tous les mots d'une phrase simultanément. Cette innovation a facilité l'entraînement de grands modèles de langage
sur des volumes massifs de données, ouvrant la voie aux modèles modernes utilisés dans la traduction automatique,
la génération de texte, le question answering et les systèmes RAG.
"""

resume = summarizer(
    texte_long,
    max_length=70,
    min_length=25,
    do_sample=False
)

print("Texte original :")
print(texte_long)

print("\nRésumé généré :")
print(resume[0]["summary_text"])

### Analyse du résultat

Le modèle de résumé automatique produit une version plus courte du texte initial.

Un bon résumé doit :

- Garder les idées principales.
- Supprimer les détails secondaires.
- Rester cohérent et compréhensible.